# AMANAH — Trusted Islamic Sources Builder

This notebook builds/validates the source bundle used by the project without changing the AMANAH model API contract.

**Priority rule:** if the hackathon scientific bundle names an approved source, use it first. The external sources here are provenance-controlled fallbacks/technical sources and are not automatically 'hackathon approved'.


In [ ]:
# 1) Environment
import hashlib, io, json, os, re, shutil, zipfile
from pathlib import Path
from urllib.parse import urljoin
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','requests>=2.32','beautifulsoup4>=4.12','pandas>=2.2'],check=True)
import requests, pandas as pd
from bs4 import BeautifulSoup
ROOT=Path('/content/AMANAH_trusted_sources'); ROOT.mkdir(parents=True,exist_ok=True)
print('Output:',ROOT)


## Qur'an Arabic canonical text — KFGQPC
Official developer package: King Fahd Glorious Qur'an Printing Complex, Unicode Uthmanic Hafs. The official developer page lists v13.0 and SHA-1 `36EA5AB0D7EA1702F17FF43F9B50924CCCD77EBF`.


In [ ]:
# 2) Download + integrity-check KFGQPC Uthmanic Hafs package
QURAN_URL='https://download.qurancomplex.gov.sa/resources_dev/UthmanicHafs_v2-0.zip'
QURAN_SHA1='36EA5AB0D7EA1702F17FF43F9B50924CCCD77EBF'.lower()
qzip=ROOT/'UthmanicHafs_v2-0.zip'
r=requests.get(QURAN_URL,timeout=180); r.raise_for_status(); qzip.write_bytes(r.content)
sha1=hashlib.sha1(qzip.read_bytes()).hexdigest(); print('SHA-1:',sha1)
assert sha1==QURAN_SHA1, f'KFGQPC checksum mismatch: {sha1}'
qdir=ROOT/'kfgqpc_hafs'; qdir.mkdir(exist_ok=True)
with zipfile.ZipFile(qzip) as z: z.extractall(qdir)
json_candidates=list(qdir.rglob('*.json'))
print('JSON files:',[p.name for p in json_candidates[:20]])


In [ ]:
# 3) Normalize the official Qur'an JSON to AMANAH's ayah-id format
def load_rows(path):
    obj=json.loads(path.read_text(encoding='utf-8-sig'))
    if isinstance(obj,list): return obj
    if isinstance(obj,dict):
        for k in ('data','ayahs','items','rows'):
            if isinstance(obj.get(k),list): return obj[k]
        if all(isinstance(v,dict) for v in obj.values()): return list(obj.values())
    raise ValueError(f'Unsupported JSON structure: {path}')
preferred=[p for p in json_candidates if 'hafsData' in p.name or 'uthmanic' in p.name.lower()]
source_json=(preferred or json_candidates)[0]
rows=load_rows(source_json)
norm=[]
for x in rows:
    s=x.get('sura_no') or x.get('sura') or x.get('sura_number')
    a=x.get('aya_no') or x.get('aya') or x.get('ayah') or x.get('aya_number')
    t=x.get('aya_text') or x.get('text') or x.get('uthmani')
    if s and a and t: norm.append({'ayah_id':f'{int(s)}:{int(a)}','surah':int(s),'ayah':int(a),'source_ar':str(t)})
assert len(norm)==6236, f'Expected 6236 Hafs ayat, got {len(norm)}'
(ROOT/'quran_kfgqpc_hafs.json').write_text(json.dumps({'source':'KFGQPC','sha1':sha1,'ayahs':norm},ensure_ascii=False,indent=2),encoding='utf-8')
print('Qur\'an canonical rows:',len(norm))


## English Qur'an reference translations — QuranEnc API
Use these as a multi-reference comparison envelope, not as a replacement for the Arabic canonical text. The API exposes translation key/version metadata and per-ayah endpoints.


In [ ]:
# 4) Fetch available English translations and select up to three explicit references
LIST_URL='https://quranenc.com/api/v1/translations/list/en?localization=en'
available=requests.get(LIST_URL,timeout=60).json()
print(pd.DataFrame(available)[['key','title','version','last_update']].to_string(index=False))
available_by_key={x['key']:x for x in available}
preferred=['english_saheeh','english_rwwad','english_hilali_khan']
selected=[k for k in preferred if k in available_by_key]
if len(selected)<3:
    selected += [k for k in available_by_key if k not in selected][:3-len(selected)]
print('Selected:',selected)
assert len(selected)>=2, 'Need at least two independent English references'


In [ ]:
# 5) Build per-ayah English reference cache from QuranEnc
refs={aid:[] for aid in [x['ayah_id'] for x in norm]}
for key in selected:
    meta=available_by_key[key]
    for sura in range(1,115):
        url=f'https://quranenc.com/api/v1/translation/sura/{key}/{sura}'
        res=requests.get(url,timeout=60); res.raise_for_status(); payload=res.json()
        items=payload.get('result',payload) if isinstance(payload,dict) else payload
        for x in items:
            aid=f"{int(x['sura'])}:{int(x['aya'])}"
            refs.setdefault(aid,[]).append({'provider':'QuranEnc','translation_key':key,'version':str(meta.get('version','unknown')),'title':meta.get('title'),'text':x.get('translation',''),'source_url':url})
bundle=[]
byid={x['ayah_id']:x for x in norm}
for aid,row in byid.items(): bundle.append({**row,'references':refs.get(aid,[])})
assert all(len(x['references'])==len(selected) for x in bundle)
(ROOT/'quran_reference_bundle_kfgqpc_quranenc.json').write_text(json.dumps({'ayahs':bundle},ensure_ascii=False),encoding='utf-8')
print('Reference bundle:',len(bundle),'ayahs ×',len(selected),'translations')


## Hadith — Dorar Al-Sunniah official Hadith API
The documented API is a search/retrieval interface. Preserve the returned source/grading information. AMANAH must abstain if a hadith cannot be traced to a source and grading.


In [ ]:
# 6) Dorar hadith API client + reproducible sample
DORAR_API='https://dorar.net/dorar_api.json'
def dorar_hadith_search(query):
    r=requests.get(DORAR_API,params={'skey':query},timeout=60); r.raise_for_status(); return r.json()
sample=dorar_hadith_search('إنما الأعمال بالنيات')
(ROOT/'hadith_dorar_sample.json').write_text(json.dumps(sample,ensure_ascii=False,indent=2),encoding='utf-8')
print('Hadith API returned keys:',list(sample)[:20])
print('IMPORTANT: use as traceable retrieval; do not invent or infer a grade.')


## Tafsir — KFGQPC Tafseer Muyassar + Dorar verification
KFGQPC's developer portal describes Tafseer Muyassar as trusted/approved by the Complex and provides developer files with a separate `aya_tafseer` field. The hackathon compliance report also cites Dorar Tafsir for the approved-reference path. No documented public Dorar Tafsir REST API was verified, so this notebook does **not** scrape Dorar.


In [ ]:
# 7) Discover/download the official KFGQPC Tafseer Muyassar developer ZIP
DEV_PAGE='https://qurancomplex.gov.sa/en/techquran/dev/'
TAFSIR_SHA1='5f533113c2f54f32eded734bb49e6a5837965722'
html=requests.get(DEV_PAGE,timeout=120,headers={'User-Agent':'Mozilla/5.0'}).text
soup=BeautifulSoup(html,'html.parser')
candidates=[]
for a in soup.find_all('a',href=True):
    href=urljoin(DEV_PAGE,a['href'])
    context=' '.join([a.get_text(' ',strip=True), a.parent.get_text(' ',strip=True) if a.parent else '']).lower()
    if href.lower().endswith('.zip') and ('tafseer' in context or 'tafsir' in context or 'moyassar' in context or 'muyassar' in context or 'تفسير' in context): candidates.append(href)
print('Discovered tafsir ZIP candidates:',candidates)
tzip=ROOT/'Tafseer_Muyassar_official.zip'
downloaded=False
for url in candidates:
    try:
        rr=requests.get(url,timeout=180); rr.raise_for_status(); tzip.write_bytes(rr.content)
        got=hashlib.sha1(tzip.read_bytes()).hexdigest()
        if got==TAFSIR_SHA1: downloaded=True; print('Verified official Tafsir ZIP:',url); break
    except Exception as e: print('candidate failed:',url,type(e).__name__)
if not downloaded:
    print('Automatic discovery was blocked. Download Tafseer Muyassar from the official developer page, upload the ZIP to /content, then set tzip to that path and rerun this cell.')
else:
    tdir=ROOT/'kfgqpc_tafseer_muyassar'; tdir.mkdir(exist_ok=True)
    with zipfile.ZipFile(tzip) as z: z.extractall(tdir)
    tjson=list(tdir.rglob('*.json')); print('Tafsir JSON files:',[p.name for p in tjson[:20]])


In [ ]:
# 8) Package everything for the team
manifest={'quran':{'provider':'KFGQPC','sha1':QURAN_SHA1,'file':'quran_kfgqpc_hafs.json'},'quran_translations':{'provider':'QuranEnc','keys':selected,'file':'quran_reference_bundle_kfgqpc_quranenc.json'},'hadith':{'provider':'Dorar','api':DORAR_API,'sample':'hadith_dorar_sample.json'},'tafsir':{'provider':'KFGQPC Tafseer Muyassar','expected_sha1':TAFSIR_SHA1,'challenge_manual_reference':'https://dorar.net/tafseer'}}
(ROOT/'SOURCE_MANIFEST.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
archive=shutil.make_archive('/content/AMANAH_trusted_sources_bundle','zip',ROOT)
print('READY:',archive)
print('Send SOURCE_MANIFEST.json with the data bundle so provenance is never lost.')
